# Modelado Predictivo y Explicabilidad ($MP_{2.5}$ — Tramo Homogéneo 2018–2023)

Este notebook reproduce de manera íntegra el entrenamiento y la evaluación del modelo final:
- **Datos depurados (`2018–2023`):** se excluye el tramo 2013–abril 2017 por corrimiento de escala en SINCA.
- **Variables explicativas (`5`):** `Estacion`, `Month`, `Temperatura_C`, `Humedad`, `Viento_ms` (sin contaminantes co-medidos).
- **Partición temporal estricta:** entrenamiento `2018–2021` y prueba futura `2022–2023`.
- **Comparación obligatoria:** Media de entrenamiento, Regresión Lineal de 1 variable (`Temperatura_C`), Regresión Lineal Múltiple y Random Forest.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

plt.style.use('seaborn-v0_8-whitegrid')

FEATURES = ['Estacion', 'Month', 'Temperatura_C', 'Humedad', 'Viento_ms']
TARGET = 'MP2_5'

df = pd.read_parquet('../data/processed/calidad_aire_limpia.parquet')
df_model = df.dropna(subset=FEATURES + [TARGET]).sort_values(['Fecha', 'Estacion']).reset_index(drop=True)

train_mask = df_model['Year'] <= 2021
df_train = df_model[train_mask].copy()
df_test = df_model[~train_mask].copy()

X_train, y_train = df_train[FEATURES], df_train[TARGET]
X_test, y_test = df_test[FEATURES], df_test[TARGET]

print(f'Total observaciones completas (3 estaciones con clima): {len(df_model):,}')
print(f'Entrenamiento (2018-2021): {len(X_train):,} filas | Prueba futura (2022-2023): {len(X_test):,} filas')
print('Estaciones en prueba:', df_test['Estacion'].value_counts().to_dict())

Total observaciones completas (3 estaciones con clima): 5,505
Entrenamiento (2018-2021): 3,870 filas | Prueba futura (2022-2023): 1,635 filas
Estaciones en prueba: {'Santiago': 551, 'Pudahuel': 549, 'Las Condes': 535}


## Entrenamiento y Tabla Comparativa contra Soluciones Base

In [2]:
def make_prep():
    return ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), ['Estacion']),
            ('num', 'passthrough', ['Month', 'Temperatura_C', 'Humedad', 'Viento_ms']),
        ],
        verbose_feature_names_out=False,
    )

# 1. Base 1: Media constante de entrenamiento
pred_media = np.full(len(y_test), y_train.mean(), dtype=float)

# 2. Base 2: Regresión lineal simple (1 variable: Temperatura_C)
lr_1var = LinearRegression().fit(X_train[['Temperatura_C']], y_train)
pred_lr1 = lr_1var.predict(X_test[['Temperatura_C']])

# 3. Regresión Lineal Múltiple
pipe_lr = Pipeline([('prep', make_prep()), ('model', LinearRegression())])
pipe_lr.fit(X_train, y_train)
pred_lr = pipe_lr.predict(X_test)

# 4. Random Forest Regressor (Modelo Final)
pipe_rf = Pipeline([
    ('prep', make_prep()),
    ('model', RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_leaf=4, random_state=42, n_jobs=-1)),
])
pipe_rf.fit(X_train, y_train)
pred_rf = pipe_rf.predict(X_test)

def calcular_metricas(y_real, y_pred, nombre):
    return {
        'Modelo': nombre,
        'MAE': round(mean_absolute_error(y_real, y_pred), 3),
        'RMSE': round(np.sqrt(mean_squared_error(y_real, y_pred)), 3),
        'MAPE (%)': round(mean_absolute_percentage_error(y_real, y_pred) * 100, 2),
        'R2': round(r2_score(y_real, y_pred), 4),
    }

metricas = pd.DataFrame([
    calcular_metricas(y_test, pred_media, 'Base 1: Media entrenamiento'),
    calcular_metricas(y_test, pred_lr1, 'Base 2: Reg. Lineal (Temperatura_C)'),
    calcular_metricas(y_test, pred_lr, 'Regresión Lineal Múltiple'),
    calcular_metricas(y_test, pred_rf, 'Random Forest Regressor (Final)'),
])
print('--- Rendimiento sobre el mismo conjunto de prueba (2022-2023) ---')
display(metricas)

--- Rendimiento sobre el mismo conjunto de prueba (2022-2023) ---


,Modelo,MAE,RMSE,MAPE (%),R2
0,Base 1: Media entrenamiento,13.175,16.980,76.18,-0.0001
1,Base 2: Reg. Lineal (Temperatura_C),10.885,15.364,63.59,0.1812
2,Regresión Lineal Múltiple,10.487,14.291,63.41,0.2916
3,Random Forest Regressor (Final),8.093,12.053,43.37,0.4961


## Gráficos de Ajuste y Residuos (Prueba 2022–2023)

In [3]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

axes[0, 0].scatter(y_test, pred_lr, alpha=0.45, color='#3b73b9')
axes[0, 0].plot([0, y_test.max()], [0, y_test.max()], 'r--', lw=1.8)
axes[0, 0].set_title('Regresión Lineal Múltiple: Real vs Predicho')
axes[0, 0].set_xlabel('MP2.5 Real (µg/m³)')
axes[0, 0].set_ylabel('Predicción (µg/m³)')

axes[0, 1].scatter(y_test, pred_rf, alpha=0.45, color='#2e7d32')
axes[0, 1].plot([0, y_test.max()], [0, y_test.max()], 'r--', lw=1.8)
axes[0, 1].set_title('Random Forest Final: Real vs Predicho')
axes[0, 1].set_xlabel('MP2.5 Real (µg/m³)')

axes[1, 0].scatter(pred_lr, y_test - pred_lr, alpha=0.45, color='#3b73b9')
axes[1, 0].axhline(0, color='r', linestyle='--', lw=1.8)
axes[1, 0].set_title('Regresión Lineal Múltiple: Residuos')
axes[1, 0].set_xlabel('Predicción (µg/m³)')
axes[1, 0].set_ylabel('Residuo (Real - Predicho)')

axes[1, 1].scatter(pred_rf, y_test - pred_rf, alpha=0.45, color='#2e7d32')
axes[1, 1].axhline(0, color='r', linestyle='--', lw=1.8)
axes[1, 1].set_title('Random Forest Final: Residuos')
axes[1, 1].set_xlabel('Predicción (µg/m³)')

plt.tight_layout()
plt.savefig('graficos_rendimiento.png', dpi=200)
plt.show()

## Explicabilidad Global (SHAP Summary Plot)

In [4]:
prep_fitted = pipe_rf.named_steps['prep']
rf_fitted = pipe_rf.named_steps['model']
X_test_enc = pd.DataFrame(
    prep_fitted.transform(X_test),
    columns=prep_fitted.get_feature_names_out(),
    index=X_test.index,
)

explainer = shap.TreeExplainer(rf_fitted)
X_test_sample = X_test_enc.sample(n=min(500, len(X_test_enc)), random_state=42)
shap_values = explainer(X_test_sample)

plt.figure(figsize=(10, 5.5))
shap.summary_plot(shap_values, X_test_sample, show=False)
plt.title('Explicabilidad Global SHAP: Impacto climático y territorial en MP2.5 (2022–2023)')
plt.tight_layout()
plt.savefig('shap_global.png', dpi=200, bbox_inches='tight')
plt.show()

## Explicabilidad Local (SHAP Waterfall: Acierto y Error)

In [5]:
errores_abs = np.abs(y_test.values - pred_rf)
pos_mejor = int(np.argmin(errores_abs))
pos_peor = int(np.argmax(errores_abs))

shap_values_local = explainer(X_test_enc.iloc[[pos_mejor, pos_peor]])

print(f'CASO 1 (ACIERTO) -> Estación: {X_test.iloc[pos_mejor]["Estacion"]} | Real: {y_test.iloc[pos_mejor]:.1f} | Predicción: {pred_rf[pos_mejor]:.1f}')
plt.figure(figsize=(8, 4.5))
shap.plots.waterfall(shap_values_local[0], show=False)
plt.title('Explicación Local SHAP: Acierto del Modelo')
plt.tight_layout()
plt.savefig('shap_acierto.png', dpi=200, bbox_inches='tight')
plt.show()

print(f'CASO 2 (ERROR ALTO) -> Estación: {X_test.iloc[pos_peor]["Estacion"]} | Real: {y_test.iloc[pos_peor]:.1f} | Predicción: {pred_rf[pos_peor]:.1f}')
plt.figure(figsize=(8, 4.5))
shap.plots.waterfall(shap_values_local[1], show=False)
plt.title('Explicación Local SHAP: Subestimación en Pico Extremo')
plt.tight_layout()
plt.savefig('shap_error.png', dpi=200, bbox_inches='tight')
plt.show()

CASO 1 (ACIERTO) -> Estación: Pudahuel | Real: 13.0 | Predicción: 13.0
CASO 2 (ERROR ALTO) -> Estación: Pudahuel | Real: 95.0 | Predicción: 13.6
